<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>


# Lecture — Model Deployment with MLflow

## Overview

This lecture shows how MLflow packages a trained model once — with its code, dependencies, and metadata — so it is ready to be served through any deployment strategy. 

This lecture covers 4 sections that build on each other:

- **A. MLflow for Deployment**. MLflow's core components and the three deployment benefits it provides — dependency and environment management, packaging of models and code, and multiple deployment options.
- **B. The MLflow Model ("Flavor")**. The standard MLflow model format — the `MLmodel` file and directory — and the concept of flavors, including the default `python_function` flavor.
- **C. Models in Unity Catalog**. The centralized model store: versioning, aliases such as `@champion` and `@challenger`, lineage, and governance through the Unity Catalog three-level namespace.
- **D. Serving and Exporting Models**. How a registered model is served for batch, streaming, and real-time inference, and how it is exported to runtimes like ONNX for edge/embedded targets.

## Learning Objectives

By the end of this lecture, you will be able to:

1. Describe MLflow's deployment features and the benefits they provide.
2. Describe the concept of model flavors and the role of the `python_function` flavor.
3. Explain how Models in Unity Catalog manages the model lifecycle with versioning and aliases.
4. Describe how a registered model is served for batch, streaming, and real-time inference, and exported for edge deployment.

## A. MLflow for Deployment

### A1. The Core Components of MLflow

**MLflow** is an open-source platform for managing the end-to-end machine learning lifecycle. It is co-developed by Databricks and comes pre-installed on Databricks Runtime for ML. Four core components carry a model from experiment to production — the last two, **Model Registry** and **Model Serving & Deployment**, are what turn a trained model into a deployable asset.

<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 880 250" role="img" style="font-family: sans-serif;">
  <title>The Core Components of MLflow</title>

  <!-- Tracking -->
  <rect x="15" y="20" width="200" height="150" rx="10" fill="#ECEFF1" stroke="#607D8B" stroke-width="1.5"/>
  <text x="115" y="48" text-anchor="middle" font-size="15" font-weight="bold" fill="#1565C0">Tracking</text>
  <line x1="45" y1="60" x2="185" y2="60" stroke="#607D8B" stroke-width="1"/>
  <text x="115" y="90" text-anchor="middle" font-size="12" fill="#1B3139">Record and query</text>
  <text x="115" y="108" text-anchor="middle" font-size="12" fill="#1B3139">experiments: code,</text>
  <text x="115" y="126" text-anchor="middle" font-size="12" fill="#1B3139">data, config, results</text>

  <!-- Models -->
  <rect x="230" y="20" width="200" height="150" rx="10" fill="#ECEFF1" stroke="#607D8B" stroke-width="1.5"/>
  <text x="330" y="48" text-anchor="middle" font-size="15" font-weight="bold" fill="#1565C0">Models</text>
  <line x1="260" y1="60" x2="400" y2="60" stroke="#607D8B" stroke-width="1"/>
  <text x="330" y="90" text-anchor="middle" font-size="12" fill="#1B3139">General model</text>
  <text x="330" y="108" text-anchor="middle" font-size="12" fill="#1B3139">format that supports</text>
  <text x="330" y="126" text-anchor="middle" font-size="12" fill="#1B3139">diverse deploy tools</text>

  <!-- Model Registry -->
  <rect x="445" y="20" width="200" height="150" rx="10" fill="#FFF3E0" stroke="#F57C00" stroke-width="2"/>
  <text x="545" y="48" text-anchor="middle" font-size="15" font-weight="bold" fill="#E65100">Model Registry</text>
  <line x1="475" y1="60" x2="615" y2="60" stroke="#F57C00" stroke-width="1"/>
  <text x="545" y="88" text-anchor="middle" font-size="12" fill="#1B3139">Centralized, governed</text>
  <text x="545" y="106" text-anchor="middle" font-size="12" fill="#1B3139">lifecycle with versions</text>
  <text x="545" y="124" text-anchor="middle" font-size="12" fill="#1B3139">&amp; aliases (in UC)</text>

  <!-- Serving & Deployment -->
  <rect x="660" y="20" width="205" height="150" rx="10" fill="#FFF3E0" stroke="#F57C00" stroke-width="2"/>
  <text x="762" y="48" text-anchor="middle" font-size="15" font-weight="bold" fill="#E65100">Serving &amp; Deployment</text>
  <line x1="690" y1="60" x2="835" y2="60" stroke="#F57C00" stroke-width="1"/>
  <text x="762" y="88" text-anchor="middle" font-size="12" fill="#1B3139">Deploy registered</text>
  <text x="762" y="106" text-anchor="middle" font-size="12" fill="#1B3139">models to batch,</text>
  <text x="762" y="124" text-anchor="middle" font-size="12" fill="#1B3139">streaming &amp; real-time</text>

  <!-- Deployment bracket -->
  <rect x="445" y="185" width="420" height="46" rx="8" fill="#FFF8E1" stroke="#F57C00" stroke-width="1.5"/>
  <text x="655" y="213" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">Model Deployment</text>

  <line x1="545" y1="170" x2="545" y2="185" stroke="#F57C00" stroke-width="1.5"/>
  <line x1="762" y1="170" x2="762" y2="185" stroke="#F57C00" stroke-width="1.5"/>
</svg>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Note</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    On Databricks, the model registry is managed by <strong>Models in Unity Catalog</strong>, which provides unified governance, lineage, and versioning for all registered models. For new projects, it is recommended to use Unity Catalog to ensure consistent model management and deployment.
  </div>
</div>

### A2. Why MLflow for Deployment

Moving a model to production is where many projects stall: the environment drifts from training, a dependency is missing, or the target platform expects a different packaging. MLflow addresses these with three deployment benefits.

<table style="border-collapse: collapse; width: 100%; max-width: 900px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; margin: 16px 0;">
  <thead>
    <tr style="background-color: #1B3139;">
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Benefit</th>
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">What it provides</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Dependency &amp; Environment Management</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Ensures the deployment environment matches the training environment, so models run consistently regardless of where they are deployed.</td>
    </tr>
    <tr>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Packaging Models and Code</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Any code and configuration are packaged with the model, so it can be deployed seamlessly without any missing components.</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Multiple Deployment Options</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">One packaged model can target many environments — built-in local serving, remote container serving (AzureML, AWS SageMaker), Kubernetes clusters, and Databricks Model Serving.</td>
    </tr>
  </tbody>
</table>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    MLflow lets you package a model <em>once</em> and deploy it <em>anywhere</em>—from local machines to cloud services and Databricks Model Serving. This flexibility ensures your model is ready for production, regardless of the target environment. The next section explains what is included in an MLflow model package and how it supports seamless deployment.
  </div>
</div>

## B. The MLflow Model ("Flavor")

### B1. A Standard Format for Packaging Models

An **MLflow Model** is a standard format for packaging machine learning models. Each model is a **directory** containing arbitrary files together with an `MLmodel` file at its root. That `MLmodel` file is the manifest — it declares which **flavors** the model can be viewed in, plus metadata such as the signature and an input example, so that MLflow deployment tools know how to load and use the model.

<div style="max-width: 1000px; margin: 20px auto 0 auto; display: flex; gap: 24px; align-items: flex-start;">
<div style="flex: 1; font-size: 18px; text-align: left;">
<ul>
<li>Each <strong>MLflow Model</strong> is a directory of files plus an  <code>MLmodel</code> file.</li>
<li>The <code>MLmodel</code> file can define <strong>multiple flavors</strong> the model can be viewed in.</li>
<li>With flavors, MLflow <strong>deployment tools</strong> can understand the model.</li>
<li>The <code>MLmodel</code> file can carry <strong>additional metadata</strong> — <code>signature</code>, <code>input_example</code>, and more.</li>
</ul>
</div>

<div style="flex: 1; background: #0B2027; border-radius: 10px; padding: 16px; overflow-x: auto;">
<div style="color: #7FD4FF; font-weight: bold; font-size: 15px; margin-bottom: 8px; text-align: center;">ml<em>flow</em> Model Flavor</div>
<div style="background: #e3f2fd; color: #111; border-radius: 7px; padding: 10px 14px; font-size: 13px; max-width: 410px; margin: 0 auto 10px auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, Menlo, monospace; font-size: 13px; color: #111; white-space: pre;">
# Directory written by
# mlflow.sklearn.save_model(model, "my_model")
my_model/
├── MLmodel
├── model.pkl
├── conda.yaml
├── python_env.yaml
└── requirements.txt
</pre>
</div>

<div style="background: #e3f2fd; color: #111; border-radius: 7px; padding: 10px 14px; font-size: 13px; max-width: 410px; margin: 0 auto 10px auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, Menlo, monospace; font-size: 13px; color: #111; white-space: pre;">
# MLmodel file
sklearn:
    sklearn_version: 1.5.1
    pickled_model: model.pkl
python_function:
    loader_module: mlflow.sklearn
</pre>
</div>

</div>

### B2. Built-in Model Flavors and `pyfunc`

A **flavor** is a way of viewing and loading a model. Most models carry a framework-specific flavor (scikit-learn, PyTorch, …) *and* the generic **Python Function (`mlflow.pyfunc`)** flavor. Pyfunc serves as the **default model interface** for MLflow Python models: any MLflow Python model is expected to be loadable as a Python function, which is what lets a single model be deployed to many targets without framework-specific serving code.

<div style="max-width: 1000px; margin: 20px auto 0 auto; display: flex; gap: 24px; align-items: flex-start;">
<div style="flex: 1; font-size: 18px; text-align: left;">

**Python Function (`mlflow.pyfunc`):**
<ul>
<li>Serves as a <strong>default model interface</strong> for MLflow Python models.</li>
<li>Any MLflow Python model is expected to be loadable as a Python function.</li>
<li>Allows you to <strong>deploy models as Python functions</strong>.</li>
<li>Includes all the information necessary to <strong>load and use a model</strong>.</li>
<li>Common functions: <code>log_model</code>, <code>save_model</code>, <code>load_model</code>, <code>predict</code>.</li>
</ul>
</div>

<div style="flex: 1; background: #0B2027; border-radius: 10px; padding: 16px;">
<div style="background: #e3f2fd; color: #111; border-radius: 7px; padding: 10px 14px; font-size: 13px; max-width: 410px; margin: 0 auto 10px auto;">
<div style="color: #111; font-weight: bold; font-size: 18px; margin-bottom: 10px;">Example built-in model flavors</div>
<ul style="margin: 0; padding-left: 20px; color: #111; font-size: 16px; line-height: 1.9;">
  <li><strong>Python function</strong></li>
  <li>Spark MLlib</li>
  <li>Scikit-learn</li>
  <li>PyTorch</li>
  <li>TensorFlow</li>
  <li>LLM models: OpenAI, LangChain, HuggingFace</li>
  <li>Custom flavors</li>
</ul>
</div>
</div>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    The <code>pyfunc</code> flavor is MLflow's universal interface for Python models. With <code>pyfunc</code>, every model—regardless of its underlying framework—can be loaded and used with the same <code>predict()</code> function. This means you can deploy models to batch, streaming, or real-time endpoints without needing framework-specific code. Pyfunc simplifies deployment and ensures consistency across all serving targets.
  </div>
</div>

## C. Models in Unity Catalog

### C1. A Centralized, Governed Model Store

The **Model Registry** is a centralized model store that manages a model's lifecycle after training. On Databricks this is **Models in Unity Catalog**, where each registered model lives in the Unity Catalog three-level namespace — `catalog.schema.model` — and inherits UC governance.

<table style="border-collapse: collapse; width: 100%; max-width: 900px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 18px; margin: 16px 0;">
  <thead>
    <tr style="background-color: #1B3139;">
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Capability</th>
      <th style="padding: 12px 16px; text-align: left; color: #ffffff; border-bottom: 3px solid #FF3621;">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Deploy &amp; organize</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">A single place to register, organize, and deploy models across the organization.</td>
    </tr>
    <tr>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Versioning</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Every registered model gets numbered versions, so you always know exactly which artifact is in use.</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Lifecycle with aliases</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Point mutable aliases such as <code>@champion</code> and <code>@challenger</code> at specific versions. In Unity Catalog, model stages are <strong>not</strong> used — aliases and the UC namespace express environment and promotion instead.</td>
    </tr>
    <tr>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Collaboration &amp; permissions</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Unity Catalog access controls govern who can read, use, or promote each model.</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Lineage</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Full lineage links a model back to the data and runs that produced it.</td>
    </tr>
    <tr>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0; font-weight: 600;">Tagging &amp; annotations</td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #e0e0e0;">Attach tags and descriptions to organize and document models.</td>
    </tr>
  </tbody>
</table>

### C2. Aliases and the UC Namespace

Aliases are named, movable pointers to a specific model version. They let deployment code reference a stable name — for example, "whatever is currently `@champion`" — while you promote new versions behind the scenes without changing the code that serves the model.

<div style="max-width: 820px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 820 230" role="img" style="font-family: sans-serif;">
  <title>Aliases pointing at model versions in Unity Catalog</title>

  <!-- Namespace label -->
  <rect x="20" y="20" width="780" height="34" rx="6" fill="#1B3139"/>
  <text x="410" y="43" text-anchor="middle" font-size="14" font-weight="bold" fill="#FFFFFF">main.ml_models.churn_model  (catalog.schema.model)</text>

  <!-- Versions -->
  <rect x="120" y="90" width="160" height="50" rx="8" fill="#ECEFF1" stroke="#607D8B" stroke-width="1.5"/>
  <text x="200" y="120" text-anchor="middle" font-size="13" fill="#1B3139">Version 1</text>

  <rect x="330" y="90" width="160" height="50" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="2"/>
  <text x="410" y="120" text-anchor="middle" font-size="13" font-weight="bold" fill="#2E7D32">Version 2</text>

  <rect x="540" y="90" width="160" height="50" rx="8" fill="#FFF3E0" stroke="#F57C00" stroke-width="2"/>
  <text x="620" y="120" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">Version 3</text>

  <!-- Aliases -->
  <rect x="330" y="175" width="160" height="38" rx="19" fill="#00A972"/>
  <text x="410" y="199" text-anchor="middle" font-size="13" font-weight="bold" fill="#FFFFFF">@champion</text>

  <rect x="540" y="175" width="160" height="38" rx="19" fill="#F57C00"/>
  <text x="620" y="199" text-anchor="middle" font-size="13" font-weight="bold" fill="#FFFFFF">@challenger</text>

  <!-- Pointers -->
  <line x1="410" y1="175" x2="410" y2="145" stroke="#2E7D32" stroke-width="2" marker-end="url(#arrowC)"/>
  <line x1="620" y1="175" x2="620" y2="145" stroke="#F57C00" stroke-width="2" marker-end="url(#arrowCo)"/>

  <defs>
    <marker id="arrowC" markerWidth="9" markerHeight="7" refX="4" refY="3.5" orient="auto">
      <polygon points="0 0, 9 3.5, 0 7" fill="#2E7D32"/>
    </marker>
    <marker id="arrowCo" markerWidth="9" markerHeight="7" refX="4" refY="3.5" orient="auto">
      <polygon points="0 0, 9 3.5, 0 7" fill="#F57C00"/>
    </marker>
  </defs>
</svg>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    <code>@champion</code> and <code>@challenger</code> are aliases in Unity Catalog that point to specific model versions. <br>
    <ul style="margin-top:8px;">
      <li><b>@champion</b> marks the version currently serving production traffic.</li>
      <li><b>@challenger</b> marks a candidate version under evaluation.</li>
    </ul>
    Promoting a new model to production is as simple as moving the <code>@champion</code> alias to a different version—no code changes needed for serving. This enables safe, seamless model updates and A/B testing.
  </div>
</div>

## D. Serving and Exporting Models

### D1. From Registry to Deployment Modes

Once a model is packaged as a flavor and registered in Unity Catalog, the *same* model can be served for any of the deployment modes from the previous lecture. The flow is always the same: train → package as a flavor → register in UC → serve.

<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 880 300" role="img" style="font-family: sans-serif;">
  <title>MLflow and Model Deployment Modes</title>

  <!-- Source data -->
  <rect x="15" y="30" width="150" height="60" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="90" y="55" text-anchor="middle" font-size="13" font-weight="bold" fill="#1B3139">Delta Lake /</text>
  <text x="90" y="72" text-anchor="middle" font-size="12" fill="#1B3139">Feature Store</text>

  <!-- Model training -->
  <rect x="15" y="130" width="150" height="60" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="90" y="150" text-anchor="middle" font-size="13" font-weight="bold" fill="#1B3139">Model training</text>
  <text x="90" y="170" text-anchor="middle" font-size="11" fill="#1B3139">scikit-learn, </text>
  <text x="90" y="180" text-anchor="middle" font-size="11" fill="#1B3139">XGBoost & TensorFlow</text>

  <!-- Flavor -->
  <rect x="205" y="130" width="140" height="60" rx="8" fill="#FFF8E1" stroke="#F57C00" stroke-width="2"/>
  <text x="275" y="155" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">Model "Flavor"</text>
  <text x="275" y="173" text-anchor="middle" font-size="12" fill="#1B3139">MLflow</text>

  <!-- Registry -->
  <rect x="385" y="130" width="140" height="60" rx="8" fill="#FFF8E1" stroke="#F57C00" stroke-width="2"/>
  <text x="455" y="150" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">Model Registry</text>
  <text x="455" y="170" text-anchor="middle" font-size="12" fill="#1B3139">in UC</text>
  <text x="455" y="185" text-anchor="middle" font-size="12" fill="#1B3139">MLflow</text>

  <!-- Serving targets: mode name (left) + service name replacing the icon (right) + description -->
  <!-- Batch -->
  <rect x="600" y="18" width="265" height="56" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="618" y="40" font-size="13" font-weight="bold" fill="#2E7D32">Batch</text>
  <text x="847" y="40" text-anchor="end" font-size="11" font-weight="bold" fill="#1565C0">Apache Spark</text>
  <text x="618" y="60" font-size="11" fill="#1B3139">Offline data processing in batches</text>

  <!-- Streaming -->
  <rect x="600" y="82" width="265" height="56" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="618" y="104" font-size="13" font-weight="bold" fill="#2E7D32">Streaming</text>
  <text x="847" y="104" text-anchor="end" font-size="11" font-weight="bold" fill="#1565C0">Spark Structured Streaming</text>
  <text x="618" y="124" font-size="11" fill="#1B3139">Processing streaming data</text>

  <!-- Real-time -->
  <rect x="600" y="146" width="265" height="56" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="618" y="168" font-size="13" font-weight="bold" fill="#2E7D32">Real-time</text>
  <text x="847" y="168" text-anchor="end" font-size="11" font-weight="bold" fill="#1565C0">Databricks Model Serving</text>
  <text x="618" y="188" font-size="11" fill="#1B3139">Real-time predictions with low latency</text>

  <!-- Edge -->
  <rect x="600" y="210" width="265" height="56" rx="8" fill="#ECEFF1" stroke="#607D8B" stroke-width="1.5"/>
  <text x="618" y="232" font-size="13" font-weight="bold" fill="#455A64">Edge (Embedded)</text>
  <text x="850" y="232" text-anchor="end" font-size="11" font-weight="bold" fill="#1565C0">Docker · TF Lite</text>
  <text x="618" y="252" font-size="11" fill="#1B3139">Inference on local / edge devices</text>

  <!-- Arrows: pipeline -->
  <line x1="90" y1="90" x2="90" y2="130" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowD1)"/>
  <line x1="165" y1="160" x2="205" y2="160" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowD1)"/>
  <line x1="345" y1="160" x2="385" y2="160" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowD1)"/>

  <!-- Arrows: fan-out from registry to serving targets -->
  <path d="M525 158 C562 158, 572 46, 600 46" stroke="#2E7D32" stroke-width="1.5" fill="none" marker-end="url(#arrowD1g)"/>
  <path d="M525 159 C565 159, 575 110, 600 110" stroke="#2E7D32" stroke-width="1.5" fill="none" marker-end="url(#arrowD1g)"/>
  <path d="M525 161 C565 161, 575 174, 600 174" stroke="#2E7D32" stroke-width="1.5" fill="none" marker-end="url(#arrowD1g)"/>
  <path d="M525 163 C562 163, 575 238, 600 238" stroke="#607D8B" stroke-width="1.5" fill="none" marker-end="url(#arrowD1gr)"/>

  <defs>
    <marker id="arrowD1" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#1B3139"/></marker>
    <marker id="arrowD1g" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#2E7D32"/></marker>
    <marker id="arrowD1gr" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#607D8B"/></marker>
  </defs>
</svg>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Note</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    On Databricks, batch and streaming inference use Spark to apply models (as UDFs or in pipelines), while real-time inference is served by <strong>Databricks Model Serving</strong>. Edge and embedded deployments run off-platform on the device, which is covered in the next section.
  </div>
</div>

### D2. Exporting Models for Edge Deployment

In some cases — such as the **edge** — models must run on specific hardware, outside the Databricks platform. These environments have distinct characteristics that drive the export decision:

- **Offline / unreliable connectivity** — the model cannot depend on a call back to a server.
- **Latency requirements** — predictions must happen locally, close to the data.
- **Security and compliance** — data may not be allowed to leave the device.

For these targets, export the MLflow model to a portable inference runtime such as **ONNX** or **TensorFlow Lite**. A model trained in one framework can then run on a wide range of deployment targets (CPU, GPU, mobile, embedded) through a single interoperable format.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 190" role="img" style="font-family: sans-serif;">
  <title>Exporting via ONNX to multiple deployment targets</title>

  <!-- Training frameworks -->
  <rect x="20" y="30" width="200" height="130" rx="8" fill="#E3F2FD" stroke="#1976D2" stroke-width="1.5"/>
  <text x="120" y="52" text-anchor="middle" font-size="15" font-weight="bold" fill="#1565C0">Training framework</text>
  <text x="120" y="82" text-anchor="middle" font-size="14" fill="#1B3139">PyTorch · Keras</text>
  <text x="120" y="100" text-anchor="middle" font-size="14" fill="#1B3139">MLNET · scikit-learn</text>

  <!-- ONNX hub -->
  <circle cx="430" cy="95" r="55" fill="#1B3139"/>
  <text x="430" y="92" text-anchor="middle" font-size="16" font-weight="bold" fill="#FFFFFF">ONNX</text>
  <text x="430" y="112" text-anchor="middle" font-size="10" fill="#7FD4FF">open &amp; interoperable</text>

  <!-- Deployment targets -->
  <rect x="640" y="30" width="200" height="130" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="740" y="52" text-anchor="middle" font-size="15" font-weight="bold" fill="#2E7D32">Deployment target</text>
  <text x="740" y="82" text-anchor="middle" font-size="13" fill="#1B3139">CPU · GPU</text>
  <text x="740" y="104" text-anchor="middle" font-size="13" fill="#1B3139">FPGA · NPU</text>
  <text x="740" y="126" text-anchor="middle" font-size="13" fill="#1B3139">mobile · embedded</text>

  <!-- Arrows -->
  <line x1="220" y1="95" x2="372" y2="95" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowE)"/>
  <line x1="488" y1="95" x2="640" y2="95" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowE)"/>

  <defs>
    <marker id="arrowE" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto"><polygon points="0 0, 9 3.5, 0 7" fill="#1B3139"/></marker>
  </defs>
</svg>
</div>

### D3. MLflow and ONNX — Example

The workflow for an edge export has three steps: **convert** the model to ONNX format, **save** it with MLflow's ONNX flavor, and **score** it with the ONNX Runtime (or convert back to a native flavor). The example below converts a Keras model and logs it, then loads and scores it.

- Convert model to ONNX format.
- Save the ONNX model as the **ONNX flavor**.
- Scoring: use **ONNX Runtime**, or convert to a native flavor.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 14px; color: #0a0a0a; white-space: pre-wrap;">
<code><strong># Convert and log model as ONNX</strong>
import onnxmltools
onnx_model = onnxmltools.convert_keras(model)
mlflow.onnx.log_model(onnx_model, name="onnx-model")
</br>
<strong># Read and score model</strong>
import onnxruntime
session = onnxruntime.InferenceSession(onnx_model.SerializeToString())
input_name = session.get_inputs()[0].name
predictions = session.run(None, {input_name: data_np.astype(np.float32)})[0]</code>
</pre>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Note</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Exporting to ONNX or TensorFlow Lite is only needed for edge and embedded deployments that run off-platform. For batch, streaming, and real-time inference on Databricks, models are served directly from Unity Catalog—no export step required. Use Unity Catalog for consistent, governed model management and deployment.
  </div>
</div>

## Conclusion

In this lecture, you covered how MLflow supports model deployment on Databricks:

1. **MLflow for Deployment** — MLflow's components (Tracking, Models, Model Registry, Serving & Deployment) manage the ML lifecycle, and its deployment benefits — dependency & environment management, packaging of models and code, and multiple deployment options — let you package a model once and deploy it anywhere.
2. **The MLflow Model ("Flavor")** — An MLflow model is a standard directory format with an `MLmodel` manifest that declares one or more flavors. The default `python_function` (pyfunc) flavor gives every Python model a common `predict()` interface, so a single model can be served by many tools without framework-specific code.
3. **Models in Unity Catalog** — The registry is a centralized, governed model store. In Unity Catalog, models live in the `catalog.schema.model` namespace with numbered versions and movable aliases such as `@champion` and `@challenger` (stages are not used), plus lineage, permissions, and tagging.
4. **Serving and Exporting Models** — The same registered model can be served for batch, streaming, and real-time inference on Databricks, while edge/embedded targets are reached by exporting to portable runtimes such as ONNX or TensorFlow Lite.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>